In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
from statsmodels.stats.diagnostic import het_breuschpagan

In [ ]:
# =============================================================================
# ÉTAPE 1 : CHARGEMENT ET NETTOYAGE
# =============================================================================
print("\n" + "="*70)
print("ÉTAPE 1 : CHARGEMENT ET NETTOYAGE")
print("="*70)

print("""
COURS : Avant toute modélisation, il faut vérifier la qualité des données.
- Valeurs manquantes : peuvent fausser les calculs.
- Doublons : biaisent les coefficients.
- Aberrations (outliers) : tirent la droite de régression à tort.
""")

df = pd.read_csv('HousingData.csv', sep=';')
print(f"Dimensions initiales : {df.shape}")

# Valeurs manquantes
print("\n--- Valeurs manquantes ---")
print(df.isnull().sum())

# Doublons
print(f"\nNombre de doublons : {df.duplicated().sum()}")
df = df.drop_duplicates()

# ADAPTATION HousingData : 20 valeurs manquantes dans CRIM, ZN, INDUS, CHAS, AGE, LSTAT
# -> imputation par la médiane (nécessaire pour le VIF et la corrélation des étapes suivantes)
for col in ['CRIM', 'ZN', 'INDUS', 'CHAS', 'AGE', 'LSTAT']:
    df[col] = df[col].fillna(df[col].median())
print(f"\nValeurs manquantes après imputation : {df.isnull().sum().sum()}")

# Détection des aberrations (IQR)
print("\n--- Détection des valeurs aberrantes (IQR) sur MEDV ---")
Q1 = df['MEDV'].quantile(0.25)
Q3 = df['MEDV'].quantile(0.75)
IQR = Q3 - Q1
borne_inf = Q1 - 1.5 * IQR
borne_sup = Q3 + 1.5 * IQR
print(f"Q1 = {Q1:.2f}, Q3 = {Q3:.2f}, IQR = {IQR:.2f}")
print(f"Bornes acceptables : [{borne_inf:.2f}, {borne_sup:.2f}]")
outliers = df[(df['MEDV'] < borne_inf) | (df['MEDV'] > borne_sup)]
print(f"Nombre d'outliers : {len(outliers)}")
print("=> On les garde (ils peuvent être légitimes), mais on les note.")

In [ ]:
# =============================================================================
# ÉTAPE 2 : STATISTIQUES DESCRIPTIVES
# =============================================================================
print("\n" + "="*70)
print("ÉTAPE 2 : STATISTIQUES DESCRIPTIVES")
print("="*70)

print("""
COURS : Avant de modéliser, on décrit chaque variable.
- Moyenne, médiane, écart-type : tendance centrale et dispersion.
- Min, max : étendue des valeurs.
- Quartiles : répartition.
""")

print(df.describe().round(2))

In [ ]:
# =============================================================================
# ÉTAPE 3 : ENCODAGE DES VARIABLES CATÉGORIELLES
# =============================================================================
print("\n" + "="*70)
print("ÉTAPE 3 : ENCODAGE DES VARIABLES CATÉGORIELLES")
print("="*70)

print("""
COURS : Les modèles mathématiques ne comprennent que les nombres.
Il faut transformer les catégories en chiffres.
- Label Encoding : ordonne les catégories (ex: Petit=1, Moyen=2, Grand=3).
- One-Hot Encoding : crée une colonne binaire par catégorie.
  -> À utiliser pour les catégories sans ordre naturel.
  -> On supprime une modalité (drop_first=True) pour éviter la
     "dummy trap" (colinéarité parfaite).
""")

# ADAPTATION HousingData : toutes les variables sont déjà numériques.
# CHAS est une variable binaire (0/1) : c'est déjà l'équivalent d'un One-Hot Encoding.
# Il n'y a pas de colonne ID à supprimer.
print("CHAS avant encodage :", df['CHAS'].unique())
df['CHAS'] = df['CHAS'].astype(int)

colonnes_categorielles = df.select_dtypes(include='object').columns.tolist()
print("Variables catégorielles à encoder :", colonnes_categorielles)
if colonnes_categorielles:
    df = pd.get_dummies(df, columns=colonnes_categorielles, drop_first=True)
print("Colonnes après encodage :", df.columns.tolist())

In [ ]:
# =============================================================================
# ÉTAPE 4 : MATRICE DE CORRÉLATION
# =============================================================================
print("\n" + "="*70)
print("ÉTAPE 4 : MATRICE DE CORRÉLATION")
print("="*70)

print("""
COURS : La corrélation mesure la relation linéaire entre deux variables.
- r proche de +1 : corrélation positive forte (les deux augmentent ensemble)
- r proche de -1 : corrélation négative forte
- r proche de 0 : pas de corrélation linéaire
""")

corr_matrix = df.corr()
print("\nMatrice de corrélation :")
print(corr_matrix.round(3))

print("\n--- Corrélations avec MEDV (triées) ---")
print(corr_matrix['MEDV'].sort_values(ascending=False).round(3))

# Heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title("Matrice de corrélation des variables")
plt.tight_layout()
plt.show()

In [ ]:

# =============================================================================
# ÉTAPE 5 : MULTICOLINÉARITÉ (VIF)
# =============================================================================

import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

print("\n" + "=" * 70)
print("ÉTAPE 5 : MULTICOLINÉARITÉ (VIF)")
print("=" * 70)

print("""
COURS : La multicolinéarité, c'est quand deux variables explicatives
sont fortement corrélées entre elles. Cela peut rendre les coefficients instables.

- VIF < 5 : généralement acceptable
- 5 <= VIF < 10 : à surveiller
- VIF >= 10 : multicolinéarité potentiellement problématique
""")

# Supprimer la variable cible MEDV
X_vif = df.drop('MEDV', axis=1)

# Ajouter une constante
X_vif = sm.add_constant(X_vif)

# Calculer le VIF
vif_data = pd.DataFrame()

vif_data["Variable"] = X_vif.columns

vif_data["VIF"] = [
    variance_inflation_factor(X_vif.values, i)
    for i in range(X_vif.shape[1])
]

# Afficher les résultats du plus grand au plus petit
print(vif_data.sort_values('VIF', ascending=False).round(2))


In [ ]:
# =============================================================================
# ÉTAPE 6 : RÉGRESSION MULTIPLE (MCO)
# =============================================================================
import pandas as pd
import statsmodels.api as sm

print("\n" + "="*70)
print("ÉTAPE 6 : RÉGRESSION MULTIPLE (MCO)")
print("="*70)

print("""
COURS : La méthode des Moindres Carrés Ordinaires (MCO) cherche les
coefficients qui minimisent la somme des carrés des résidus.
Formule matricielle : β_hat = (X^T X)^(-1) X^T y
""")

# 1. Séparer variables explicatives et variable cible
X = df.drop('MEDV', axis=1)
y = df['MEDV']

# 2. Encoder les variables catégorielles (drop_first évite la colinéarité parfaite)
X = pd.get_dummies(X, drop_first=True).astype(float)
y = y.astype(float)

# 3. Supprimer les lignes avec NaN ou inf, en gardant X et y alignés
X = X.replace([float('inf'), float('-inf')], pd.NA)
data = pd.concat([X, y], axis=1).dropna()
X = data.drop('MEDV', axis=1).astype(float)
y = data['MEDV'].astype(float)

# 4. Ajouter la constante
X = sm.add_constant(X)

# 5. Estimation du modèle
model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
# =============================================================================
# ÉTAPE 7 : DIAGNOSTICS DES RÉSIDUS
# =============================================================================
print("\n" + "="*70)
print("ÉTAPE 7 : DIAGNOSTICS DES RÉSIDUS")
print("="*70)

print("""
COURS : Les résidus (e = y - y_hat) doivent respecter 3 conditions :
1. Moyenne nulle (vérifié automatiquement par MCO)
2. Variance constante (homoscédasticité)
3. Distribution normale
On utilise des graphiques et des tests statistiques.
""")

residus = model.resid
valeurs_ajustees = model.fittedvalues

# Graphique 1 : Résidus vs Ajustées
plt.figure(figsize=(10, 5))
plt.scatter(valeurs_ajustees, residus, alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Valeurs ajustées (prédictions)")
plt.ylabel("Résidus")
plt.title("Résidus vs Valeurs ajustées (Homoscédasticité)")
plt.show()

# Graphique 2 : QQ-plot
plt.figure(figsize=(6, 6))
stats.probplot(residus, dist="norm", plot=plt)
plt.title("QQ-plot des résidus (Normalité)")
plt.show()

# Graphique 3 : Histogramme
plt.figure(figsize=(8, 5))
sns.histplot(residus, kde=True, bins=30)
plt.title("Distribution des résidus")
plt.show()

# Tests
print("\n--- Test de normalité (Shapiro-Wilk) ---")
stat, p_value = stats.shapiro(residus)
print(f"Statistique = {stat:.4f}, p-value = {p_value:.4e}")
if p_value > 0.05:
    print("=> Résidus normaux (p > 0.05).")
else:
    print("=> Résidus NON normaux (p < 0.05).")

print("\n--- Test d'hétéroscédasticité (Breusch-Pagan) ---")
bp_test = het_breuschpagan(residus, X)
print(f"LM Statistic = {bp_test[0]:.4f}, p-value = {bp_test[1]:.4f}")
if bp_test[1] > 0.05:
    print("=> Homoscédasticité (variance constante).")
else:
    print("=> Hétéroscédasticité détectée.")

In [ ]:
# =============================================================================
# ÉTAPE 8 : INTERPRÉTATION FINALE
# =============================================================================
print("\n" + "="*70)
print("ÉTAPE 8 : INTERPRÉTATION FINALE")
print("="*70)

print("""
COURS : On interprète les coefficients comme suit :
- Chaque coefficient βi représente la variation de Y quand Xi augmente
  d'une unité, TOUTES CHOSES ÉGALES PAR AILLEURS.
- p-value < 0.05 : variable significative.
- R² : % de variance expliquée.
""")

print(f"\nR² = {model.rsquared:.4f} -> Le modèle explique {model.rsquared*100:.2f}% de la variance du prix médian (MEDV).")
print(f"R² ajusté = {model.rsquared_adj:.4f}")
print(f"F-statistic = {model.fvalue:.2f} (p-value = {model.f_pvalue:.2e})")

print("\nCoefficients significatifs (p < 0.05) :")
for var in X.columns:
    if var != 'const':
        coef = model.params[var]
        pval = model.pvalues[var]
        if pval < 0.05:
            print(f"  - {var} : {coef:+.2f} (p = {pval:.4e})")

In [ ]:
# =============================================================================
# ÉTAPE 10 : TEST PAR SAISIE UTILISATEUR
# =============================================================================
print("\n" + "="*70)
print("ÉTAPE 10 : TEST INTERACTIF - PRÉDICTION PAR SAISIE")
print("="*70)

print("""
COURS : Ce test permet de saisir un profil manuellement et de voir
la prédiction du modèle en temps réel. C'est un excellent moyen de
comprendre l'impact de chaque variable sur le prix médian des logements (MEDV).
""")

def saisir_valeur(message, type_val, condition=None, message_erreur=None, valeur_defaut=None):
    """
    Demande une valeur à l'utilisateur avec validation.

    - message : texte affiché
    - type_val : 'float' ou 'int'
    - condition : fonction de validation (ex: lambda x: x in [0, 1])
    - message_erreur : message si invalide
    - valeur_defaut : si l'utilisateur appuie sur Entrée
    """
    while True:
        try:
            saisie = input(message)
            if saisie == "" and valeur_defaut is not None:
                return valeur_defaut
            val = float(saisie) if type_val == 'float' else int(saisie)
            if condition is not None and not condition(val):
                print(f"ERREUR : {message_erreur}")
                continue
            return val
        except ValueError:
            print(f"ERREUR : Veuillez saisir un nombre valide.")


def saisir_profil():
    """Demande à l'utilisateur de saisir les valeurs du profil avec validation."""
    print("\n--- SAISIE DU PROFIL (QUARTIER) ---")

    crim = saisir_valeur(
        "CRIM - taux de criminalité (ex: 0.25) : ", 'float',
        condition=lambda x: x >= 0,
        message_erreur="CRIM doit être positif."
    )

    zn = saisir_valeur(
        "ZN - % de grands terrains résidentiels, 0-100 (ex: 0) : ", 'float',
        condition=lambda x: 0 <= x <= 100,
        message_erreur="ZN doit être entre 0 et 100."
    )

    indus = saisir_valeur(
        "INDUS - % de surfaces non commerciales, 0-30 (ex: 9.7) : ", 'float',
        condition=lambda x: 0 <= x <= 30,
        message_erreur="INDUS doit être entre 0 et 30."
    )

    chas = saisir_valeur(
        "CHAS - le quartier borde la rivière ? (0 = non, 1 = oui) : ", 'int',
        condition=lambda x: x in [0, 1],
        message_erreur="CHAS doit être 0 ou 1."
    )

    nox = saisir_valeur(
        "NOX - concentration en oxydes d'azote, 0-1 (ex: 0.54) : ", 'float',
        condition=lambda x: 0 < x <= 1,
        message_erreur="NOX doit être entre 0 et 1."
    )

    rm = saisir_valeur(
        "RM - nombre moyen de pièces, 3-9 (ex: 6.2) : ", 'float',
        condition=lambda x: 3 <= x <= 9,
        message_erreur="RM doit être entre 3 et 9."
    )

    age = saisir_valeur(
        "AGE - % de logements construits avant 1940, 0-100 (ex: 77) : ", 'float',
        condition=lambda x: 0 <= x <= 100,
        message_erreur="AGE doit être entre 0 et 100."
    )

    dis = saisir_valeur(
        "DIS - distance aux centres d'emploi (ex: 3.2) : ", 'float',
        condition=lambda x: x > 0,
        message_erreur="DIS doit être strictement positif."
    )

    rad = saisir_valeur(
        "RAD - accessibilité aux autoroutes, 1-24 (ex: 5) : ", 'int',
        condition=lambda x: 1 <= x <= 24,
        message_erreur="RAD doit être entre 1 et 24."
    )

    tax = saisir_valeur(
        "TAX - taux de taxe foncière pour 10 000 $ (ex: 330) : ", 'float',
        condition=lambda x: x > 0,
        message_erreur="TAX doit être strictement positif."
    )

    ptratio = saisir_valeur(
        "PTRATIO - ratio élèves/enseignants (ex: 19) : ", 'float',
        condition=lambda x: 0 < x <= 30,
        message_erreur="PTRATIO doit être entre 0 et 30."
    )

    b = saisir_valeur(
        "B - indice B, 0-400 (ex: 391) : ", 'float',
        condition=lambda x: 0 <= x <= 400,
        message_erreur="B doit être entre 0 et 400."
    )

    lstat = saisir_valeur(
        "LSTAT - % de population à faible statut, 0-100 (ex: 11) : ", 'float',
        condition=lambda x: 0 <= x <= 100,
        message_erreur="LSTAT doit être entre 0 et 100."
    )

    return {
        'CRIM': crim,
        'ZN': zn,
        'INDUS': indus,
        'CHAS': chas,
        'NOX': nox,
        'RM': rm,
        'AGE': age,
        'DIS': dis,
        'RAD': rad,
        'TAX': tax,
        'PTRATIO': ptratio,
        'B': b,
        'LSTAT': lstat
    }

def predire_par_saisie(model, X_columns, profil_dict):
    """Prédit le prix médian (MEDV) avec un profil saisi."""
    # Créer un DataFrame avec toutes les colonnes de X
    profil = pd.DataFrame(columns=X_columns)

    # Remplir les colonnes
    for col in X_columns:
        if col == 'const':
            profil.loc[0, col] = 1
        elif col in profil_dict:
            profil.loc[0, col] = profil_dict[col]
        else:
            profil.loc[0, col] = 0

    return model.predict(profil.astype(float))[0]


# --- Boucle de test ---
while True:
    profil = saisir_profil()

    if profil is None:
        print("Réessayez.\n")
        continue

    # Prédiction
    prix_predit = predire_par_saisie(model, X.columns.tolist(), profil)

    # Affichage du résultat
    print("\n" + "="*50)
    print("RÉSULTAT DE LA PRÉDICTION")
    print("="*50)
    print(f"Profil saisi :")
    print(f"  - CRIM (criminalité)     : {profil['CRIM']}")
    print(f"  - ZN                     : {profil['ZN']} %")
    print(f"  - INDUS                  : {profil['INDUS']} %")
    print(f"  - CHAS (bord de rivière) : {profil['CHAS']}")
    print(f"  - NOX (pollution)        : {profil['NOX']}")
    print(f"  - RM (pièces)            : {profil['RM']}")
    print(f"  - AGE (avant 1940)       : {profil['AGE']} %")
    print(f"  - DIS                    : {profil['DIS']}")
    print(f"  - RAD                    : {profil['RAD']}")
    print(f"  - TAX                    : {profil['TAX']}")
    print(f"  - PTRATIO                : {profil['PTRATIO']}")
    print(f"  - B                      : {profil['B']}")
    print(f"  - LSTAT                  : {profil['LSTAT']} %")
    print(f"\n=> PRIX MÉDIAN PRÉDIT (MEDV) : {prix_predit:,.2f} k$ (milliers de dollars)")
    print("="*50)

    # Continuer ?
    reponse = input("\nVoulez-vous tester un autre profil ? (o/n) : ").lower()
    if reponse != 'o':
        print("Fin du test interactif.")
        break